In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def abc_group(percent):
    if percent < 81:
        return 'A'
    elif percent < 96:
        return 'B'
    else:
        return 'C'

df_issued = pd.read_csv('Выпущенные.csv')
df_active = pd.read_parquet('Активные.parquet')
df_active['Активная карта'] = (((df_active['Приход'] > 0) | (df_active['Расход'] > 0)).groupby(df_active['ID карты']).transform('any').map({True: 'Да', False: 'Нет'}))

# ABC-анализ филиалов по объёму выпуска
abc_issued = df_issued.groupby('Филиал', as_index=False)['Количество карт'].sum()
abc_issued = abc_issued.sort_values('Количество карт', ascending=False)
abc_issued['Доля'] = (abc_issued['Количество карт'] / abc_issued['Количество карт'].sum() * 100).round(2)
abc_issued['Накопленная доля'] = (abc_issued['Количество карт'].cumsum() / abc_issued['Количество карт'].sum() * 100).round(2)
abc_issued['Сегмент выпуск'] = abc_issued['Накопленная доля'].apply(abc_group)

# ABC-анализ филиалов по количеству активных карт
abc_active = df_active[df_active['Активная карта'] == 'Да'].groupby('Филиал', as_index=False)['ID карты'].nunique()
abc_active = abc_active.rename(columns={'ID карты': 'Активно'})
abc_active = abc_active.sort_values('Активно', ascending=False)
abc_active['Доля активных'] = (abc_active['Активно'] / abc_active['Активно'].sum() * 100).round(2)
abc_active['Накопленная доля активных'] = (abc_active['Активно'].cumsum() / abc_active['Активно'].sum() * 100).round(2)
abc_active['ABC активных'] = abc_active['Накопленная доля активных'].apply(abc_group)

# Объединение результатов и формирование итоговых групп
abc_issued = abc_issued.rename(columns={
    'Количество карт': 'Выпущено',
    'Доля': 'Доля выпуска',
    'Накопленная доля': 'Накопленная доля выпуска',
    'Сегмент выпуск': 'ABC выпуска'
})

abc = abc_issued.merge(abc_active, on='Филиал', how='left')
abc['Выпущено'] = abc['Выпущено'].map(lambda x: f'{x:,.0f}'.replace(',', ' '))
abc['Активно'] = abc['Активно'].map(lambda x: f'{x:,.0f}'.replace(',', ' '))
abc['Группа'] = abc['ABC выпуска'] + abc['ABC активных']

# Цветовое оформление итоговой таблицы
colors = {'AA': '#5B9BD5', 'BB': '#9DC3E6', 'CC': '#D9EAF7', 'BA': '#D9EAD3', 'CA': '#D9EAD3', 'CB': '#D9EAD3', 'AB': '#FFF2CC', 'AC': '#FFF2CC', 'BC': '#FFF2CC'}

def color_row(row):
    bg = colors.get(row['Группа'], '#FFFFFF')
    text = 'inherit'
    return [f'background-color: {bg}; color: {text};'] * len(row)

from IPython.display import HTML, display

# Форматируем доли и накопленные доли
abc_display = abc.copy()

for col in ['Доля выпуска', 'Накопленная доля выпуска', 'Доля активных', 'Накопленная доля активных']:
    abc_display[col] = abc_display[col].map(lambda x: f'{x:.2f}')

html = '<table style="border-collapse: collapse; text-align: left;">'

html += '<tr>'
for col in abc_display.columns:
    html += f'<th style="padding: 6px 10px; border: 1px solid #ccc;">{col}</th>'
html += '</tr>'

for _, row in abc_display.iterrows():
    bg = colors.get(row['Группа'], '#FFFFFF')
    html += f'<tr style="background-color: {bg};">'

    for value in row:
        html += f'<td style="padding: 6px 10px; border: 1px solid #ccc;">{value}</td>'

    html += '</tr>'

html += '</table>'

display(HTML(html))

Филиал,Выпущено,Доля выпуска,Накопленная доля выпуска,ABC выпуска,Активно,Доля активных,Накопленная доля активных,ABC активных,Группа
Кадамжайский филиал,47 176,10.79,10.79,A,33 209,11.67,23.58,A,AA
"Филиал ""Манас""",44 118,10.09,20.88,A,33 886,11.91,11.91,A,AA
Бишкекский филиал,40 964,9.37,30.25,A,22 200,7.80,63.54,A,AA
Чуйский филиал,38 629,8.83,39.08,A,23 185,8.15,31.73,A,AA
Ошский филиал,35 561,8.13,47.22,A,22 606,7.95,47.81,A,AA
"Филиал ""Ош-Датка""",32 962,7.54,54.75,A,23 155,8.14,39.87,A,AA
Узгенский филиал,32 304,7.39,62.14,A,22 557,7.93,55.74,A,AA
Центральный филиал,21 141,4.84,66.98,A,13 932,4.90,68.44,A,AA
Каракольский филиал,20 939,4.79,71.77,A,13 005,4.57,77.81,A,AA
Кербенский филиал,20 449,4.68,76.44,A,13 660,4.80,73.24,A,AA
